
### JAB-Hessian sensitivity estimation & adaptive precision allocation



**Contents**
1. GPTQ core (shared quantizer, same as Student A's)
2. Calibration data capture (X and target attention output A(X))
3. Attention-aware joint loss (MSE + optional KL)
4. Hutchinson trace estimator
5. Greedy sensitivity-per-cost allocator
6. Shared utilities (calibration batches, perplexity)
7. Load GPT-2 + validate the from-scratch attention loss against the real model
8. JAB-Hessian: per-block sensitivity scores
9. Full pipeline: uniform GPTQ baseline
10. Full pipeline: JAB-Hessian adaptive allocation
11. Compare results


In [19]:
!pip install -q torch transformers datasets

In [20]:
import math
import random
import itertools

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd.functional import hessian as exact_hessian
import torch.autograd as autograd

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


## 1. GPTQ core


In [21]:
def collect_hessian_via_hook(model: torch.nn.Module, module: torch.nn.Module,
                              calibration_batches, device) -> torch.Tensor:
    """
    Registers a forward pre-hook on `module` (e.g. one block's attn.c_attn)
    to capture its input activations, runs `calibration_batches` through the
    *whole model* in no_grad mode, and returns the accumulated Hessian
    H = 2 X^T X for that layer.

    `calibration_batches` should be an iterable of input_ids tensors of shape
    (batch, seq_len), already on `device`.

    Returns: H, a (d_in, d_in) double-precision tensor.
    """
    d_in = module.weight.shape[0]  # Conv1D weight is (in_features, out_features)
    H = torch.zeros(d_in, d_in, dtype=torch.float64, device=device)
    n_samples = [0]

    def _hook(mod, inputs):
        x = inputs[0].detach()
        x = x.reshape(-1, x.shape[-1]).to(torch.float64)  # (tokens, d_in)
        H.add_(2.0 * x.T @ x)
        n_samples[0] += x.shape[0]

    handle = module.register_forward_pre_hook(_hook)
    try:
        model.eval()
        with torch.no_grad():
            for input_ids in calibration_batches:
                model(input_ids.to(device))
    finally:
        handle.remove()

    if n_samples[0] > 0:
        H /= n_samples[0]
    return H


def _quantize_to_grid(w_col: torch.Tensor, scale: torch.Tensor, bits: int) -> torch.Tensor:
    """
    Symmetric per-output-row fake quantization of a single input-column
    (shape: d_out) using a fixed per-row scale (shape: d_out) computed
    up front from the original weight statistics.
    """
    qmax = 2 ** (bits - 1) - 1
    q = torch.clamp(torch.round(w_col / scale), -qmax, qmax)
    return q * scale


@torch.no_grad()
def gptq_quantize_layer(weight_in_out: torch.Tensor, H: torch.Tensor, bits: int = 4,
                         damp_percent: float = 0.01, group_size: int = None,
                         act_order: bool = True) -> torch.Tensor:
    """
    Quantizes a weight matrix in the (d_in, d_out) "Conv1D" convention
    (GPT-2 style: forward is x @ weight) using the GPTQ algorithm.

    weight_in_out: (d_in, d_out) float tensor -- e.g. c_attn.weight.data
    H: (d_in, d_in) Hessian from collect_hessian_via_hook
    bits: target bit-width
    damp_percent: Hessian damping factor for numerical stability (GPTQ default ~0.01)
    group_size: if set (e.g. 128), computes a separate per-row scale for each
        contiguous group of `group_size` input columns instead of one scale
        for the whole row. Finer granularity -> lower quantization error,
        at essentially no extra cost. None = one scale per row (previous
        default behavior).
    act_order: if True, quantizes columns in order of decreasing Hessian
        diagonal (most "sensitive" columns first, while the most
        compensation budget is still available) instead of naive
        left-to-right order. Standard GPTQ accuracy improvement.

    Returns the fake-quantized weight, same shape, and also writes it into
    weight_in_out in place.
    """
    device = weight_in_out.device
    W = weight_in_out.detach().clone().to(torch.float64).T.contiguous()  # (d_out, d_in)
    d_out, d_in = W.shape

    # Damp and (optionally) reorder the Hessian before inverting.
    H = H.clone()
    mean_diag = H.diagonal().mean()
    H += damp_percent * mean_diag * torch.eye(d_in, dtype=torch.float64, device=device)

    if act_order:
        perm = torch.argsort(torch.diag(H), descending=True)
        invperm = torch.argsort(perm)
        W = W[:, perm]
        H = H[perm][:, perm]
    else:
        perm = torch.arange(d_in, device=device)
        invperm = perm

    H_inv = torch.linalg.inv(H)

    # Per-(row, group) scale, computed once from the original weights
    # (in the possibly-permuted column order) before any quantization.
    qmax = 2 ** (bits - 1) - 1
    gs = group_size if group_size is not None else d_in
    n_groups = (d_in + gs - 1) // gs
    scale = torch.zeros(d_out, n_groups, dtype=torch.float64, device=device)
    for g in range(n_groups):
        start, end = g * gs, min((g + 1) * gs, d_in)
        scale[:, g] = (W[:, start:end].abs().amax(dim=1) / qmax).clamp(min=1e-8)

    for i in range(d_in):
        row_scale = scale[:, i // gs]
        w_col = W[:, i]
        q_col = _quantize_to_grid(w_col, row_scale, bits)
        err = (w_col - q_col) / H_inv[i, i]
        if i + 1 < d_in:
            W[:, i + 1:] -= torch.outer(err, H_inv[i, i + 1:])
        W[:, i] = q_col

    if act_order:
        W = W[:, invperm]

    W_final = W.T.contiguous().to(weight_in_out.dtype)  # back to (d_in, d_out)
    weight_in_out.copy_(W_final)
    return W_final

## 2. Calibration data capture


In [22]:
class AttentionOutputCapture:
    """
    Captures attention output A(X) from each block, used for MSE loss.
    """
    def __init__(self, model):
        self.outputs = {}
        self.handles = []

        for idx, block in enumerate(model.transformer.h):
            # Hook before c_proj (this is A(X) before output projection)
            handle = block.attn.c_proj.register_forward_pre_hook(self._make_hook(idx))
            self.handles.append(handle)

    def _make_hook(self, idx):
        def hook(module, inputs):
            # input[0] is A(X) - the attention output
            self.outputs[idx] = inputs[0].detach().clone()
        return hook

    def remove(self):
        for h in self.handles:
            h.remove()


class ActivationCapture:
    """
    Captures input activations (X) to each block's attention.
    """
    def __init__(self, model):
        self.activations = {}
        self.handles = []

        for idx, block in enumerate(model.transformer.h):
            # Hook before c_attn (this is X, the input to attention)
            handle = block.attn.c_attn.register_forward_pre_hook(self._make_hook(idx))
            self.handles.append(handle)

    def _make_hook(self, idx):
        def hook(module, input):
            # input[0] is X
            self.activations[idx] = input[0].detach().clone()
        return hook

    def remove(self):
        for h in self.handles:
            h.remove()


def get_calibration_data(model, calibration_batch, device):
    """
    Get all calibration data for one forward pass.
    Returns X, target_A, target_attn (None for now - MSE only).
    """
    # Create captures
    act_capture = ActivationCapture(model)
    out_capture = AttentionOutputCapture(model)

    # Run model
    model.eval()
    with torch.no_grad():
        model(calibration_batch.to(device))

    # Get data
    X = act_capture.activations
    target_A = out_capture.outputs
    target_attn = None  # MSE only for now (KL can be added later)

    # Clean up
    act_capture.remove()
    out_capture.remove()

    return X, target_A, target_attn


def get_calibration_data_for_block(model, calibration_batch, device, block_idx):
    """
    Get calibration data for a specific block.
    """
    X_dict, target_A_dict, target_attn_dict = get_calibration_data(
        model, calibration_batch, device
    )

    return (
        X_dict[block_idx],
        target_A_dict[block_idx],
        None  # This will be None #target_attn_dict[block_idx]
    )

## 3. Attention-aware joint loss
`L = ||A(X) - A_hat(X)||^2 (+ lambda * KL(attention maps), optional)` -- a from-scratch, differentiable multi-head attention computation (with causal masking) as a pure function of a flattened `[W_Q | W_K | W_V]` parameter vector, which is exactly what the Hutchinson estimator needs to differentiate through.

In [23]:
def reshape_weights(w_flat, n_embd):
    """
    Convert flattened weights back to Q, K, V matrices.
    Inputs:
        w_flat: Flattened [W_Q, W_K, W_V]
        n_embd: Model dimension (768 for GPT-2 small)
    Outputs:
        W_Q, W_K, W_V: Each of shape (n_embd, n_embd)
    """
    # Each matrix has n_embd * n_embd parameters
    size = n_embd * n_embd

    # Split into 3 parts
    W_Q = w_flat[0:size].reshape(n_embd, n_embd)
    W_K = w_flat[size:2*size].reshape(n_embd, n_embd)
    W_V = w_flat[2*size:3*size].reshape(n_embd, n_embd)

    return W_Q, W_K, W_V

def compute_attention(W_Q, W_K, W_V, X, n_head = 12, b_Q=None, b_K=None, b_V=None):
    """
    Compute attention output and attention weights.
    Inputs:
        W_Q, W_K, W_V: Weight matrices (n_embd, n_embd)
        X: Input activations (batch, seq_len, n_embd)
    Outputs:
        A_hat: Attention output (batch, seq_len, n_embd)
        attn_weights: Attention weights (batch, seq_len, seq_len)
    """
    B, T, n_embd = X.shape #batch size, sequence length, 768
    d_head = n_embd // n_head #per-head dimension
    # Compute Q, K, V
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    if b_Q is not None:
        Q = Q + b_Q
        K = K + b_K
        V = V + b_V

    Q = Q.view(B, T, n_head, d_head).transpose(1, 2)
    K = K.view(B, T, n_head, d_head).transpose(1, 2)
    V = V.view(B, T, n_head, d_head).transpose(1, 2)
    # Scaled dot-product attention
    #d_k = Q.shape[-1]
    #scale = torch.sqrt(torch.tensor(d_k, dtype=torch.float32, device=Q.device))
    scale = math.sqrt(d_head)
    scores = Q @ K.transpose(-2, -1) / scale

    causal_mask = torch.tril(torch.ones(T, T, device=X.device, dtype=torch.bool))
    scores = scores.masked_fill(~causal_mask, float("-inf"))
    # Attention weights
    attn_weights = torch.softmax(scores, dim=-1)
    # Attention output (weighted sum of values)
    #A_hat = attn_weights @ V
    A_hat = (attn_weights @ V).transpose(1, 2).contiguous().view(B, T, n_embd)

    return A_hat, attn_weights

def mse_loss(w_flat, X, target_A, n_embd, b_Q=None, b_K=None, b_V=None):
    """
    L_mse = ||A(X) - A_hat(X)||^2
    This is the primary loss function.
    """
    # Reshape and compute attention
    W_Q, W_K, W_V = reshape_weights(w_flat, n_embd)
    A_hat, _ = compute_attention(W_Q, W_K, W_V, X, b_Q=b_Q, b_K=b_K, b_V=b_V)
    # MSE loss
    return F.mse_loss(A_hat, target_A)

def kl_loss(w_flat, X, target_attn, n_embd, b_Q=None, b_K=None, b_V=None):
    """
    L_kl = KL(attention_weights || target_attention_weights)
    """
    # Reshape and compute attention
    W_Q, W_K, W_V = reshape_weights(w_flat, n_embd)
    _, attn_weights = compute_attention(W_Q, W_K, W_V, X, b_Q=b_Q, b_K=b_K, b_V=b_V)
    # KL divergence: KL(P || Q) = sum(P * log(P / Q))
    # P = target_attn, Q = attn_weights
    log_q = torch.log(attn_weights + 1e-8)  # Add epsilon for stability

    return F.kl_div(log_q, target_attn, reduction='batchmean') #batchmean = sum(KL) / batch_size (like in Q-BERT & APTQ)

def attention_loss(w_flat, X, target_A, n_embd, target_attn=None, lambda_kl=0.1, b_Q=None, b_K=None, b_V=None):
    """
    If target_attn is None: Returns MSE only O.W. :Returns MSE + lambda_kl * KL
    """
    # Always compute MSE
    loss = mse_loss(w_flat, X, target_A, n_embd, b_Q=b_Q, b_K=b_K, b_V=b_V)
    # Add KL if target attention weights are provided
    if target_attn is not None:
        kl = kl_loss(w_flat, X, target_attn, n_embd, b_Q=b_Q, b_K=b_K, b_V=b_V)
        loss = loss + lambda_kl * kl

    return loss

### Sanity check: `attention_loss.py` unit tests
All run on synthetic data (no GPT-2 needed) -- verifies shapes, gradient flow, numerical stability, and that MSE-only and MSE+KL agree when KL's weight/target is absent.

In [24]:
def test_reshape_weights():
    print("\n=== Test 1: Reshape Weights ===")
    n_embd = 768
    total = 3 * n_embd * n_embd
    # Create random flat weights
    w_flat = torch.randn(total)
    # Reshape
    W_Q, W_K, W_V = reshape_weights(w_flat, n_embd)
    # Check shapes
    assert W_Q.shape == (n_embd, n_embd)
    assert W_K.shape == (n_embd, n_embd)
    assert W_V.shape == (n_embd, n_embd)
    # Check we can reconstruct
    reconstructed = torch.cat([W_Q.flatten(), W_K.flatten(), W_V.flatten()])
    assert torch.allclose(w_flat, reconstructed, atol=1e-6)
    print("Reshape works correctly")
    print(f"   Input shape: {w_flat.shape}")
    print(f"   W_Q shape: {W_Q.shape}")

def test_compute_attention():
    print("\n=== Test 2: Compute Attention ===")
    n_embd = 768
    batch = 2
    seq = 4
    # Create random inputs
    W_Q = torch.randn(n_embd, n_embd)
    W_K = torch.randn(n_embd, n_embd)
    W_V = torch.randn(n_embd, n_embd)
    X = torch.randn(batch, seq, n_embd)
    # Compute attention
    A_hat, attn_weights = compute_attention(W_Q, W_K, W_V, X)
    n_head = 12  # default in compute_attention
    # Check shapes
    assert A_hat.shape == (batch, seq, n_embd)
    assert attn_weights.shape == (batch, n_head, seq, seq)
    # Check attention weights sum to 1 (per row)
    sums = attn_weights.sum(dim=-1)
    assert torch.allclose(sums, torch.ones(batch, n_head, seq), atol=1e-5)
    print("Attention computation works")
    print(f"   A_hat shape: {A_hat.shape}")
    print(f"   Attn weights sum: {sums[0, 0, 0].item():.6f}")

def test_mse_loss():
    print("\n=== Test 3: MSE Loss ===")
    n_embd = 768
    batch = 2
    seq = 4
    # Create random data
    w_flat = torch.randn(3 * n_embd * n_embd)
    X = torch.randn(batch, seq, n_embd)
    target_A = torch.randn(batch, seq, n_embd)
    # Compute loss
    loss = attention_loss(w_flat, X, target_A, n_embd)
    # Check it's a scalar and non-negative
    assert loss.shape == ()  # Scalar
    assert loss.item() >= 0  # Non-negative
    print(f"MSE loss works")
    print(f"   Loss: {loss.item():.6f}")

def test_gradients():
    print("\n=== Test 4: Gradient Flow ===")
    n_embd = 768
    batch = 2
    seq = 4
    # Create data with requires_grad
    w_flat = torch.randn(3 * n_embd * n_embd, requires_grad=True)
    X = torch.randn(batch, seq, n_embd)
    target_A = torch.randn(batch, seq, n_embd)
    # Forward pass
    loss = attention_loss(w_flat, X, target_A, n_embd)
    # Backward pass
    loss.backward()
    # Check gradients
    assert w_flat.grad is not None
    assert w_flat.grad.shape == w_flat.shape
    assert torch.any(w_flat.grad != 0)  # Non-zero gradients
    print(f"Gradients flow correctly")
    print(f"   Gradient shape: {w_flat.grad.shape}")
    print(f"   Gradient norm: {w_flat.grad.norm().item():.6f}")

def test_kl_loss():
    print("\n=== Test 5: KL Divergence Loss ===")
    n_embd = 768
    batch = 2
    seq = 4
    # Create random data
    w_flat = torch.randn(3 * n_embd * n_embd)
    X = torch.randn(batch, seq, n_embd)
    target_A = torch.randn(batch, seq, n_embd)
    # Create random target attention weights (valid probability distribution)
    n_head = 12
    target_attn = torch.softmax(torch.randn(batch, n_head, seq, seq), dim=-1)
    # MSE only
    loss_mse = attention_loss(w_flat, X, target_A, n_embd)
    # MSE + KL
    loss_combined = attention_loss(w_flat, X, target_A, n_embd, target_attn, lambda_kl=0.1)
    # Combined loss should be >= MSE loss (since KL >= 0)
    assert loss_combined.item() >= loss_mse.item()
    print(f"KL loss works")
    print(f"   MSE only: {loss_mse.item():.6f}")
    print(f"   MSE + KL: {loss_combined.item():.6f}")

def test_different_shapes():
    print("\n=== Test 6: Different Shapes ===")
    n_embd = 768
    # Test different batch sizes
    for batch in [1, 2, 4]:
        seq = 4
        w_flat = torch.randn(3 * n_embd * n_embd)
        X = torch.randn(batch, seq, n_embd)
        target_A = torch.randn(batch, seq, n_embd)

        loss = attention_loss(w_flat, X, target_A, n_embd)
        print(f"   Batch {batch}: loss = {loss.item():.6f}")
    # Test different sequence lengths
    for seq in [1, 4, 8]:
        batch = 2
        w_flat = torch.randn(3 * n_embd * n_embd)
        X = torch.randn(batch, seq, n_embd)
        target_A = torch.randn(batch, seq, n_embd)

        loss = attention_loss(w_flat, X, target_A, n_embd)
        print(f"   Seq {seq}: loss = {loss.item():.6f}")

    print("Works with different shapes")

def test_numerical_stability():
    print("\n=== Test 7: Numerical Stability ===")
    n_embd = 768
    batch = 2
    seq = 4
    # Test with large values
    w_flat = torch.randn(3 * n_embd * n_embd) * 1000
    X = torch.randn(batch, seq, n_embd) * 1000
    target_A = torch.randn(batch, seq, n_embd) * 1000
    loss = attention_loss(w_flat, X, target_A, n_embd)
    # Should not be NaN or Inf
    assert not torch.isnan(loss)
    assert not torch.isinf(loss)
    print(f"Stable with extreme values: {loss.item():.6f}")

def test_attention_loss_consistency():
    print("\n=== Test 8: Loss Consistency ===")
    n_embd = 768
    batch = 2
    seq = 4
    w_flat = torch.randn(3 * n_embd * n_embd)
    X = torch.randn(batch, seq, n_embd)
    target_A = torch.randn(batch, seq, n_embd)
    # Method 1: Direct mse_loss
    loss1 = mse_loss(w_flat, X, target_A, n_embd)
    # Method 2: attention_loss without KL
    loss2 = attention_loss(w_flat, X, target_A, n_embd)
    # Method 3: attention_loss with target_attn=None
    loss3 = attention_loss(w_flat, X, target_A, n_embd, target_attn=None)
    assert torch.allclose(loss1, loss2, atol=1e-6)
    assert torch.allclose(loss1, loss3, atol=1e-6)
    print(f"All methods give consistent results")
    print(f"   mse_loss: {loss1.item():.6f}")
    print(f"   attention_loss: {loss2.item():.6f}")

if __name__ == "__main__":
    print("TESTING ATTENTION_LOSS.PY (No GPT-2 Required)")

    test_reshape_weights()
    test_compute_attention()
    test_mse_loss()
    test_gradients()
    test_kl_loss()
    test_different_shapes()
    test_numerical_stability()
    test_attention_loss_consistency()
    print("ALL TESTS PASSED!")
    print("\nThe loss function is ready to use with GPT-2.")

TESTING ATTENTION_LOSS.PY (No GPT-2 Required)

=== Test 1: Reshape Weights ===
Reshape works correctly
   Input shape: torch.Size([1769472])
   W_Q shape: torch.Size([768, 768])

=== Test 2: Compute Attention ===
Attention computation works
   A_hat shape: torch.Size([2, 4, 768])
   Attn weights sum: 1.000000

=== Test 3: MSE Loss ===
MSE loss works
   Loss: 741.790100

=== Test 4: Gradient Flow ===
Gradients flow correctly
   Gradient shape: torch.Size([1769472])
   Gradient norm: 1600.354248

=== Test 5: KL Divergence Loss ===
KL loss works
   MSE only: 712.823242
   MSE + KL: 776.198242

=== Test 6: Different Shapes ===
   Batch 1: loss = 728.624329
   Batch 2: loss = 857.942139
   Batch 4: loss = 782.265930
   Seq 1: loss = 778.439758
   Seq 4: loss = 806.623352
   Seq 8: loss = 791.038086
Works with different shapes

=== Test 7: Numerical Stability ===
Stable with extreme values: 722739446939648.000000

=== Test 8: Loss Consistency ===
All methods give consistent results
   mse_lo

## 4. Hutchinson trace estimator
`trace(H) ~= (1/n) * sum(v_i^T H v_i)` for Rademacher vectors `v_i`, using two backward passes (double-backward) to get exact Hessian-vector products without ever forming the full Hessian.

In [25]:
def hessian_vector_product(loss_fn, params, vector, retain_graph = True):
    #first order grad
    grad = autograd.grad(loss_fn(params), params, create_graph=True, retain_graph=True)[0] #retain_graph = true to keep the graph for second grad, do not release it!
    #second order grad
    hvp = autograd.grad(grad, params, grad_outputs=vector, retain_graph=True)[0]
    return hvp

def hutchinson_trace_estimator(loss_fn, params, samples=50): #number of iterations mentioned in HAWQ-V2 article
    #trace(H) ≈ (1/n) * Σ(v_i^T * H * v_i)
    if not params.requires_grad:
        params.requires_grad_(True)

    device = params.device
    estimated_trace = 0.0

    for _ in range(samples):
        #Rademacher Vector(mentioned in HAWQ-V2)
        vec = torch.randint(0, 2, params.shape, device=device) * 2 -1
        vec = vec.float()
        hvp_result = hessian_vector_product(loss_fn, params, vec)
        estimated_trace += torch.dot(vec.flatten(), hvp_result.flatten())

    return estimated_trace/samples

### Sanity check: Hutchinson estimate vs. exact Hessian trace (toy layer)
On a tiny `nn.Linear(5, 3)` we can afford to compute the *exact* Hessian via `torch.autograd.functional.hessian` and compare it to the Hutchinson estimate directly.

In [26]:
def test_hutchinson():

    #toy layer
    layer = nn.Linear(5, 3)
    x = torch.randn(4, 5) #input data
    y = torch.randn(4, 3) #target data

    weights = list(layer.parameters())[0].clone()
    weights_flat = weights.flatten()
    weights_flat.requires_grad_(True)
    print(f"shape of weights: {weights.shape}")
    print(f"number of weights: {weights.numel()}")

    def loss_fn(w):
        w_reshaped = w.reshape(3, 5)
        output = torch.matmul(x, w_reshaped.T) + layer.bias
        return nn.MSELoss()(output, y)

    test_loss = loss_fn(weights_flat)
    print(f"Test loss: {test_loss.item():.6f}")

    H_matrix = exact_hessian(loss_fn, weights_flat)
    H_2d = H_matrix.reshape(weights_flat.numel(), weights_flat.numel())
    exact_trace = torch.trace(H_2d)
    print(f"exact trace: {exact_trace:0.6f}")

    hutchinson_trace = hutchinson_trace_estimator(loss_fn=loss_fn, params=weights_flat, samples=300)
    print(f"Hutchinson estimated trace: {hutchinson_trace:0.6f}")

    print("\n3. Comparing results...")
    error = abs(hutchinson_trace - exact_trace)
    print(f"Exact trace:      {exact_trace:0.6f}")
    print(f"Hutchinson:       {hutchinson_trace:0.6f}")
    print(f"Error:            {error:0.6f}")

    if error < 0.1:
        print("Test passed!")
    else:
        print(f"Error too large: {error:0.6f}")

torch.manual_seed(0)  # fixed seed: this is a Monte Carlo estimator,
# without a seed the pass/fail outcome is flaky run-to-run (verified:
# a real toy-problem run gave error 0.05 once and 0.32 another time,
# purely from random-draw variance, not a code issue)
test_hutchinson()

shape of weights: torch.Size([3, 5])
number of weights: 15
Test loss: 1.258623
exact trace: 6.869429
Hutchinson estimated trace: 6.871145

3. Comparing results...
Exact trace:      6.869429
Hutchinson:       6.871145
Error:            0.001717
Test passed!


## 5. Greedy sensitivity-per-cost allocator
Starts every block at the highest bit-width, then repeatedly downgrades whichever block loses the least accuracy per unit of budget freed, until the budget is met -- then spends any leftover budget on the best available upgrades.

In [27]:
#Greedy sensitivity-per-cost bit-width allocator
import random
import itertools

BIT_WIDTHS = [2, 3, 4, 8, 16] #2, 16 is deleted for now!

def generate_synthetic_scores(block_names, seed=0):
    """
    Returns a dictionary of sensitivity scores per bit width per each block.
    """
    rng = random.Random(seed)
    scores = {}

    for name in block_names:
        fragility = rng.uniform(0.5, 3.0)
        sensitivity_by_bits = {}

        for bits in BIT_WIDTHS:
            # more bits -> less sensitivity
            sensitivity = fragility / (bits ** 1.5)
            sensitivity_by_bits[bits] = sensitivity

        scores[name] = sensitivity_by_bits

    return scores

def cost(bits, param_count=1.0):
    """
    Memory cost of storing a block at a given bit-width.
    """
    return bits * param_count

def greedy_allocate(scores, budget):
    """
    scores: dict of {block_name: {bits: sensitivity}}
    budget: max total cost allowed
      1. Give every block the HIGHEST bit-width (best accuracy, most cost).
      2. While we're over budget: find the one downgrade (one block, one
         step down in bits) that saves the most cost per unit of accuracy
         lost, and apply it. Repeat.
      3. If we still have leftover budget afterward, try upgrading blocks
         back up wherever it's affordable and helps the most.
    """
    # start at max precision
    current_bits = {name: max(BIT_WIDTHS) for name in scores}

    def total_cost():
        return sum(cost(current_bits[name]) for name in scores)

    def total_sensitivity():
        return sum(scores[name][current_bits[name]] for name in scores)

    # downgrade loop
    while total_cost() > budget:
        best_block = None
        best_new_bits = None
        best_ratio = None  # (sensitivity) / (cost)

        for name in scores:
            bits_now = current_bits[name]
            lower_choices = [b for b in BIT_WIDTHS if b < bits_now]
            if not lower_choices:
                continue  # already at the lowest possible bit-width

            next_bits = max(lower_choices)
            cost_saved = cost(bits_now) - cost(next_bits)
            sensitivity_added = scores[name][next_bits] - scores[name][bits_now]

            ratio = sensitivity_added / cost_saved

            if best_ratio is None or ratio < best_ratio:
                best_ratio = ratio
                best_block = name
                best_new_bits = next_bits

        if best_block is None:
            break  # can't downgrade anything further

        current_bits[best_block] = best_new_bits

    # spend remaining budget on the best upgrade available
    made_an_upgrade = True
    while made_an_upgrade:
        made_an_upgrade = False
        best_block = None
        best_new_bits = None
        best_ratio = None  # (sensitivity) / (extra cost)

        for name in scores:
            bits_now = current_bits[name]
            higher_choices = [b for b in BIT_WIDTHS if b > bits_now]
            if not higher_choices:
                continue

            next_bits = min(higher_choices)
            extra_cost = cost(next_bits) - cost(bits_now)

            if total_cost() + extra_cost > budget:
                continue  # can't afford it

            sensitivity_saved = scores[name][bits_now] - scores[name][next_bits]
            ratio = sensitivity_saved / extra_cost

            if best_ratio is None or ratio > best_ratio:
                best_ratio = ratio
                best_block = name
                best_new_bits = next_bits

        if best_block is not None:
            current_bits[best_block] = best_new_bits
            made_an_upgrade = True

    return current_bits, total_cost(), total_sensitivity()

def brute_force_optimal(scores, budget):
    """
    Tries every possible combination of bit-widths and keeps the best one that fits the budget. Only usable for a small number of blocks.
    """
    names = list(scores.keys())
    choices_per_block = [BIT_WIDTHS] * len(names)

    best_assignment = None
    best_cost = None
    best_sensitivity = float("inf")

    for combo in itertools.product(*choices_per_block):
        total_c = sum(cost(bits) for bits in combo)
        if total_c > budget:
            continue

        total_s = sum(scores[names[i]][combo[i]] for i in range(len(names)))

        if total_s < best_sensitivity:
            best_sensitivity = total_s
            best_cost = total_c
            best_assignment = dict(zip(names, combo))

    return best_assignment, best_cost, best_sensitivity

### Sanity check: greedy allocator vs. brute-force optimal (synthetic scores)
On a small 4-block slice we can afford to check every combination exactly, to confirm the greedy heuristic isn't leaving accuracy on the table.

In [28]:
# 12 blocks = W_Q, W_K, W_V for 4 layers
block_names = [f"L{layer}_{proj}" for layer in range(4) for proj in ("WQ", "WK", "WV")]
scores = generate_synthetic_scores(block_names, seed=42)

max_cost = sum(max(BIT_WIDTHS) for _ in block_names)
min_cost = sum(min(BIT_WIDTHS) for _ in block_names)
budget = min_cost + 0.35 * (max_cost - min_cost)

print(f"{len(block_names)} blocks | min_cost={min_cost} max_cost={max_cost} budget={budget:.1f}\n")

assignment, cost_used, sensitivity = greedy_allocate(scores, budget)
print("Greedy allocation:")
for name in block_names:
    print(f"  {name:10s} -> {assignment[name]:2d} bits")
print(f"\nTotal cost: {cost_used:.1f} (budget {budget:.1f})")
print(f"Total sensitivity: {sensitivity:.4f}")
# validate against brute force on a small 4-block slice
small_names = block_names[:4]
small_scores = {name: scores[name] for name in small_names}
small_max = sum(max(BIT_WIDTHS) for _ in small_names)
small_min = sum(min(BIT_WIDTHS) for _ in small_names)
small_budget = small_min + 0.35 * (small_max - small_min)

greedy_assign, greedy_cost, greedy_sens = greedy_allocate(small_scores, small_budget)
optimal_assign, optimal_cost, optimal_sens = brute_force_optimal(small_scores, small_budget)

print("\n--- Validation on 4 blocks ---")
print(f"Greedy : cost={greedy_cost:.1f} sensitivity={greedy_sens:.4f} -> {greedy_assign}")
print(f"Optimal: cost={optimal_cost:.1f} sensitivity={optimal_sens:.4f} -> {optimal_assign}")

12 blocks | min_cost=24 max_cost=192 budget=82.8

Greedy allocation:
  L0_WQ      ->  8 bits
  L0_WK      ->  4 bits
  L0_WV      ->  8 bits
  L1_WQ      ->  8 bits
  L1_WK      ->  8 bits
  L1_WV      ->  8 bits
  L2_WQ      ->  8 bits
  L2_WK      ->  4 bits
  L2_WV      ->  8 bits
  L3_WQ      ->  4 bits
  L3_WK      ->  4 bits
  L3_WV      ->  8 bits

Total cost: 80.0 (budget 82.8)
Total sensitivity: 1.0223

--- Validation on 4 blocks ---
Greedy : cost=24.0 sensitivity=0.3478 -> {'L0_WQ': 8, 'L0_WK': 4, 'L0_WV': 8, 'L1_WQ': 4}
Optimal: cost=27.0 sensitivity=0.3002 -> {'L0_WQ': 8, 'L0_WK': 3, 'L0_WV': 8, 'L1_WQ': 8}


## 6. Shared utilities
Calibration-batch construction and sliding-window perplexity evaluation (same as Student A's).

In [29]:
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

def build_calibration_batches(tokenizer, n_samples=128, seq_len=512):
    """
    Pulls `n_samples` chunks of `seq_len` tokens each from WikiText-2 train,
    as a list of (1, seq_len) input_id tensors.
    """
    raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    text = "\n\n".join(t for t in raw["text"] if t.strip())
    ids = tokenizer(text, return_tensors="pt").input_ids[0]

    batches = []
    stride = seq_len
    for i in range(n_samples):
        start = i * stride
        if start + seq_len > ids.shape[0]:
            break
        chunk = ids[start:start + seq_len].unsqueeze(0)
        batches.append(chunk)
    return batches


@torch.no_grad()
def evaluate_perplexity(model, tokenizer, max_length=1024, stride=512):
    """
    Sliding-window perplexity on WikiText-2 test.
    """
    raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join(t for t in raw["text"] if t.strip())
    ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    seq_len = ids.shape[1]

    model.eval()
    nll_sum = 0.0
    n_tokens = 0
    prev_end = 0

    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        input_ids = ids[:, begin:end]
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        out = model(input_ids, labels=target_ids)
        nll_sum += out.loss.item() * trg_len
        n_tokens += trg_len

        prev_end = end
        if end == seq_len:
            break

    return math.exp(nll_sum / n_tokens)

## 7. Load GPT-2 and validate the from-scratch attention loss
Before trusting JAB-Hessian scores computed from `attention_loss.py`'s reimplemented attention, confirm it reproduces GPT-2's *real* attention output almost exactly.

In [30]:
model = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model.eval()
n_embd = model.config.n_embd
print(f"Model dimension: {n_embd}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model dimension: 768


In [31]:
# a small real batch
text = "The quick brown fox jumps over the lazy dog. " * 20
batch = tokenizer(text, return_tensors="pt", truncation=True, max_length=64).input_ids.to(DEVICE)

block_idx = 0
X, target_A, _ = get_calibration_data_for_block(model, batch, DEVICE, block_idx)

# reconstruct w_flat from the REAL, unquantized weights of this block
block = model.transformer.h[block_idx]
W = block.attn.c_attn.weight.data  # (768, 2304) = [W_Q | W_K | W_V]
W_Q, W_K, W_V = W.split(n_embd, dim=1)
w_flat = torch.cat([W_Q.flatten(), W_K.flatten(), W_V.flatten()])

# reconstruct the real bias too -- same split pattern as the weights
bias = block.attn.c_attn.bias.data  # (2304,)
b_Q, b_K, b_V = bias.split(n_embd)

with torch.no_grad():
    err = mse_loss(w_flat, X, target_A, n_embd, b_Q=b_Q, b_K=b_K, b_V=b_V)
    W_Q_r, W_K_r, W_V_r = reshape_weights(w_flat, n_embd)
    A_hat, attn_weights = compute_attention(W_Q_r, W_K_r, W_V_r, X, b_Q=b_Q, b_K=b_K, b_V=b_V)

mse_value = err.item()
target_scale = (target_A**2).mean().item()

print(f"attn_weights shape (expect (batch, 12, T, T)): {tuple(attn_weights.shape)}")
print(f"MSE between real attention output and compute_attention's output: {mse_value:.10f}")
print(f"Scale reference -- mean squared magnitude of target_A itself: {target_scale:.10f}")
print(f"Ratio (MSE / target scale): {mse_value / target_scale:.10f}")

if mse_value / target_scale < 1e-4:
    print("\nPASS -- compute_attention reconstructs the real attention output almost exactly.")
else:
    print("\nFAIL -- still a meaningful gap; compute_attention does not match GPT-2's real attention yet.")

attn_weights shape (expect (batch, 12, T, T)): (1, 12, 64, 64)
MSE between real attention output and compute_attention's output: 0.0000000000
Scale reference -- mean squared magnitude of target_A itself: 0.0253827553
Ratio (MSE / target scale): 0.0000000000

PASS -- compute_attention reconstructs the real attention output almost exactly.


## 8. JAB-Hessian: per-block sensitivity scores
Wires the Hutchinson estimator to the real attention-output loss (Section 3) to get one sensitivity score per block, converts it to a per-bit-width sensitivity table, and runs the greedy allocator (Section 5) against it.

In [32]:
def compute_jab_trace_from_data(model, block_idx, X, target_A, device, n_embd, samples=30):
    """
    Compute sensitivity score per attention block.
    inputs:
        model: GPT-2 model
        block_idx: Which layer
        batch: One batch of input tokens
        n_embd: Model dimension
        samples: Number of Hutchinson samples
    """
    # 1. Get weights of this block's Q, K, V
    block = model.transformer.h[block_idx]
    W = block.attn.c_attn.weight.data  # Shape: (768, 2304) = [W_Q | W_K | W_V]
    # Split into Q, K, V and flatten them into one vector
    W_Q, W_K, W_V = W.split(n_embd, dim=1)
    w_flat = torch.cat([W_Q.flatten(), W_K.flatten(), W_V.flatten()])
    w_flat.requires_grad_(True)  # for Hessian computation
    # 2. Get calibration data (X = input, target_A = attention output)
    bias = block.attn.c_attn.bias.data
    b_Q, b_K, b_V = bias.split(n_embd)
    # 3. Loss function
    def loss_fn(params):
        return attention_loss(params, X, target_A, n_embd, b_Q=b_Q, b_K=b_K, b_V=b_V)
    # 4. Compute Hessian trace using Hutchinson estimator
    trace = hutchinson_trace_estimator(loss_fn, w_flat, samples=samples)
    return trace.item()

def compute_all_jab_scores(model, batches, device, n_embd, samples=30, n_batches_to_use=4):
    """
    Compute sensitivity scores for ALL attention blocks and returns it in format of a dictionary.
    """
    n_blocks = len(model.transformer.h)
    use_batches = batches[:n_batches_to_use]
    all_traces = {f"block_{i}_QKV": [] for i in range(n_blocks)}

    print(f"\nComputing JAB scores for {n_blocks} blocks "
          f"(averaged over {len(use_batches)} batches, {samples} Hutchinson samples each)...")

    for b in use_batches:
        # ONE forward pass captures X/target_A for every block at once
        X_dict, target_A_dict, _ = get_calibration_data(model, b, device)
        for idx in range(n_blocks):
            trace = compute_jab_trace_from_data(
                model, idx, X_dict[idx], target_A_dict[idx], device, n_embd, samples
            )
            all_traces[f"block_{idx}_QKV"].append(trace)

    scores = {}
    for block_name, traces in all_traces.items():
        trace = sum(traces) / len(traces)
        scores[block_name] = trace
        level = "HIGH" if trace > 100 else "MEDIUM" if trace > 10 else "LOW"
        print(f"  {block_name}: {trace:.4f}  [{level} sensitivity]  "
              f"(min={min(traces):.4f}, max={max(traces):.4f})")

    print("Done!\n")
    return scores

def scores_to_allocator_format(jab_scores):
    """
    Convert JAB scores to format expected by greedy allocator.
    """
    allocator_input = {}
    bit_widths = [2, 3, 4, 8, 16]#[2,

    for block_name, trace in jab_scores.items():
        sensitivity_by_bits = {}
        for bits in bit_widths:
            sensitivity_by_bits[bits] = trace / (bits ** 1.5)
        allocator_input[block_name] = sensitivity_by_bits

    return allocator_input

def measure_weight_perturbation(original_W, H, bits):
    """
    HAWQ-V2 style: ||Q(W) - W||_F^2, measured directly in weight space.
    """
    w_copy = original_W.clone()
    gptq_quantize_layer(w_copy, H, bits=bits, group_size=128, act_order=True)
    perturbation = torch.norm(w_copy - original_W, p='fro') ** 2
    return perturbation.item()


def scores_to_allocator_format_hawqv2(jab_scores, model, calibration_batches, device, bit_widths=None):
    """
    HAWQ-V2 style: Omega_i(bits) = trace_i * ||Q(W_i) - W_i||_F^2,
    instead of the guessed trace / bits^1.5.
    """
    if bit_widths is None:
        bit_widths = [2, 3, 4, 8, 16]

    allocator_input = {}
    for block_name, trace in jab_scores.items():
        block_idx = int(block_name.split("_")[1])          # extract the index from "block_0_QKV" -> 0
        block = model.transformer.h[block_idx]
        c_attn = block.attn.c_attn
        original_W = c_attn.weight.data

        H = collect_hessian_via_hook(model, c_attn, calibration_batches, device)

        sensitivity_by_bits = {}
        for bits in bit_widths:
            perturbation = measure_weight_perturbation(original_W, H, bits)
            sensitivity_by_bits[bits] = trace * perturbation
        allocator_input[block_name] = sensitivity_by_bits

        print(f"  {block_name}: " + ", ".join(f"{b}bit={v:.4e}" for b, v in sensitivity_by_bits.items()))

    return allocator_input


def run_jab_allocation(model, batches, device, n_embd, target_avg_bits=4.0, samples=30, n_batches_to_use=16):

    """
    1. Compute JAB scores for all blocks
    2. Convert to allocator format (HAWQ-V2 style, measured perturbation)
    3. Run greedy allocation
    4. Return bit assignment in format of a dictionary
    """
    print("JAB-Hessian Adaptive Allocation")
    # Compute JAB scores
    print("\nComputing JAB scores...")
    jab_scores = compute_all_jab_scores(model, batches, device, n_embd, samples, n_batches_to_use)
    # Convert format
    print("Preparing for allocator...")
    allocator_input = scores_to_allocator_format_hawqv2(jab_scores, model, batches, device)
    # Set budget
    n_blocks = len(allocator_input)
    budget = target_avg_bits * n_blocks
    print(f"Budget: {budget:.1f} bits ({target_avg_bits} avg for {n_blocks} blocks)")
    # Run allocation
    print("\nRunning greedy allocation...")
    assignment, cost_used, sensitivity = greedy_allocate(allocator_input, budget)
    # results
    print("Results")
    print(f"Total cost: {cost_used:.1f} bits")
    print(f"Average bits: {cost_used / n_blocks:.2f}")
    print(f"Total sensitivity: {sensitivity:.6f}")
    print("\nPer-block allocation:")
    for block_name, bits in assignment.items():
        print(f"    {block_name}: {bits} bits")

    return assignment

## 9. Full pipeline: uniform GPTQ baseline
Quantizes every block's `c_attn` to a flat 4 bits -- the R7 comparison point for adaptive allocation. Both this and Section 10 reuse the *same* calibration batches, so the comparison is apples-to-apples.

In [33]:
print("Building the shared calibration set (used for both experiments below)...")
calibration_batches = build_calibration_batches(tokenizer, n_samples=128, seq_len=512)#32,128
print(f"{len(calibration_batches)} calibration batches ready.\n")

print("Loading a fresh full-precision GPT-2 for the uniform baseline...")
model_uniform = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
model_uniform.eval()

print("Quantizing ALL blocks to 4 bits uniformly...")
for idx in range(len(model_uniform.transformer.h)):
    block = model_uniform.transformer.h[idx]
    c_attn = block.attn.c_attn
    print(f"  Block {idx}: quantizing to 4 bits...")
    H = collect_hessian_via_hook(model_uniform, c_attn, calibration_batches, DEVICE)
    gptq_quantize_layer(c_attn.weight.data, H, bits=4, group_size=128, act_order=True)

print("\nEvaluating perplexity...")
ppl_uniform = evaluate_perplexity(model_uniform, tokenizer)
print(f"\nUniform 4-bit perplexity: {ppl_uniform:.3f}")

Building the shared calibration set (used for both experiments below)...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2415650 > 1024). Running this sequence through the model will result in indexing errors


128 calibration batches ready.

Loading a fresh full-precision GPT-2 for the uniform baseline...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Quantizing ALL blocks to 4 bits uniformly...
  Block 0: quantizing to 4 bits...
  Block 1: quantizing to 4 bits...
  Block 2: quantizing to 4 bits...
  Block 3: quantizing to 4 bits...
  Block 4: quantizing to 4 bits...
  Block 5: quantizing to 4 bits...
  Block 6: quantizing to 4 bits...
  Block 7: quantizing to 4 bits...
  Block 8: quantizing to 4 bits...
  Block 9: quantizing to 4 bits...
  Block 10: quantizing to 4 bits...
  Block 11: quantizing to 4 bits...

Evaluating perplexity...

Uniform 4-bit perplexity: 26.225


## 10. Full pipeline: JAB-Hessian adaptive allocation
Computes JAB scores, allocates bits under the same average-bit budget (4.0) as the uniform baseline, applies it, and evaluates perplexity -- using the *same* calibration batches as Section 9.

In [34]:
def apply_allocation(model, assignment, calibration_batches, device):
    print("\nApplying allocation")
    for idx in range(len(model.transformer.h)):
        block_name = f"block_{idx}_QKV"
        if block_name in assignment:
            bits = assignment[block_name]
            block = model.transformer.h[idx]
            c_attn = block.attn.c_attn
            H = collect_hessian_via_hook(model, c_attn, calibration_batches, device)
            gptq_quantize_layer(c_attn.weight.data, H, bits=bits, group_size=128, act_order=True)
            print(f"  Block {idx}: {bits} bits")
    print("Allocation done.")
    return model

In [35]:
print("Loading a fresh full-precision GPT-2 for adaptive allocation...")
model_adaptive = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
model_adaptive.eval()

print("\nComputing JAB-Hessian scores and allocating bits...")
torch.manual_seed(42)
assignment = run_jab_allocation(model=model_adaptive, batches=calibration_batches,
                                 device=DEVICE, n_embd=n_embd, target_avg_bits=4.0, samples=20)

print("\nApplying the allocation...")
model_adaptive = apply_allocation(model_adaptive, assignment, calibration_batches, DEVICE)

print("\nEvaluating perplexity...")
ppl_adaptive = evaluate_perplexity(model_adaptive, tokenizer)
print(f"\nAdaptive (JAB-Hessian) perplexity: {ppl_adaptive:.3f}")

Loading a fresh full-precision GPT-2 for adaptive allocation...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Computing JAB-Hessian scores and allocating bits...
JAB-Hessian Adaptive Allocation

Computing JAB scores...

Computing JAB scores for 12 blocks (averaged over 16 batches, 20 Hutchinson samples each)...
  block_0_QKV: 8.8141  [LOW sensitivity]  (min=8.5435, max=9.1991)
  block_1_QKV: 27.3088  [MEDIUM sensitivity]  (min=26.5417, max=28.2310)
  block_2_QKV: 97.5410  [MEDIUM sensitivity]  (min=94.5012, max=101.7118)
  block_3_QKV: 102.0859  [HIGH sensitivity]  (min=96.5775, max=107.4939)
  block_4_QKV: 68.3341  [MEDIUM sensitivity]  (min=66.5661, max=70.5521)
  block_5_QKV: 61.7517  [MEDIUM sensitivity]  (min=57.0077, max=68.5626)
  block_6_QKV: 51.4415  [MEDIUM sensitivity]  (min=46.9889, max=57.5877)
  block_7_QKV: 43.6951  [MEDIUM sensitivity]  (min=39.8768, max=47.2856)
  block_8_QKV: 43.9092  [MEDIUM sensitivity]  (min=40.1584, max=49.8791)
  block_9_QKV: 37.4549  [MEDIUM sensitivity]  (min=34.0669, max=41.5738)
  block_10_QKV: 40.3476  [MEDIUM sensitivity]  (min=36.1168, max=46.818

## 11. Compare results

In [36]:
print(f"Uniform  4-bit GPTQ perplexity : {ppl_uniform:.3f}")
print(f"Adaptive JAB-Hessian perplexity : {ppl_adaptive:.3f}")
print(f"Difference                      : {ppl_adaptive - ppl_uniform:+.3f}")

Uniform  4-bit GPTQ perplexity : 26.225
Adaptive JAB-Hessian perplexity : 26.225
Difference                      : +0.000
